# ShiftBench v2: fix detection, then re-measure (GPU, one click)

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 1.5–2.5 hours. Keep the tab open.

What this run changes from v1 (which found AUROC 0.41–0.61):
* 4 frames/s instead of 2, 60 teleop demos instead of 50, up to 40 rollouts per policy, 80 test episodes
* two new failure scores next to world-model surprise: **trajectory deviation** (live) and **end state** (after the run)
* three tasks instead of one: eye drops → basket, ring insertion, block stacking
* results zipped to your Google Drive at the end

In [ ]:
!git clone -q https://github.com/kishoren18-hue/shiftbench.git 2>/dev/null; %cd /content/shiftbench
!git pull -q && pip -q install -e . "transformers>=4.53" av opencv-python-headless pyarrow imageio-ffmpeg tabulate huggingface_hub
import torch; assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU first"; print(torch.cuda.get_device_name(0))

In [ ]:
TASKS = {"eyedrops": "Put the eye drops into the basket",
         "ring": "Insert the colourful ring into the central wooden peg",
         "blocks": "Stack the colouful blocks on top of each other"}
COMMON = ('--fps 4 --max_per_policy 40 --max_teleop 60 --max_test_B 80 --alpha 0.1 --k 15 '
          '--shifts nominal,light_dim_2,cam_bump_1,cam_bump_2,occlusion '
          '--encoders vjepa2,dinov2_s,siglip2_b,clip_b32,resnet18 '
          '--variants full,noaction,nochange,trajdev,endstate --data_dir /content/armnet --cache /content/sb_cache')
for name, task in TASKS.items():
    print("=" * 20, name)
    !python scripts/offline_suite.py --task "{task}" {COMMON} --out /content/out_{name} 2>&1 | grep -v "it/s]" | tail -40
    !python scripts/summarize_suite.py /content/out_{name}

In [ ]:
# Quick read: AUROC per score variant (higher = better failure detection; 0.5 = chance)
import pandas as pd, glob
rows = []
for name in TASKS:
    for f in glob.glob(f"/content/out_{name}/table_B_*_*.csv"):
        t = pd.read_csv(f); t = t[(t.method == "fixed") & (t.condition == "nominal")]
        if len(t): rows.append({"task": name, "encoder": t.encoder.iloc[0], "variant": t.variant.iloc[0], "auroc": t.auroc_raw.iloc[0]})
pd.DataFrame(rows).pivot_table(index=["task", "encoder"], columns="variant", values="auroc").round(3)

In [ ]:
# Replay demo with the best-detecting V-JEPA 2 model on the first task, then save everything to Drive
!python scripts/replay_demo.py --data_dir /content/armnet --detector /content/out_eyedrops/detector_vjepa2 --encoder vjepa2 --fps 4 \
    --episodes_csv /content/out_eyedrops/episodes.csv --traces /content/out_eyedrops/traces.parquet --out /content/out_eyedrops/demo --n 2
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/shiftbench_v2 && cd /content && zip -qr /content/drive/MyDrive/shiftbench_v2/results_v2.zip out_* -x "*/detector_*/*"
print("Saved to Drive: MyDrive/shiftbench_v2/results_v2.zip — tell Claude it's there.")